# L5b Lab: Maximum-Flow Sensitivity and Bottlenecks

> **Learning objectives**
>
> - Establish a validated maximum-flow baseline.
> - Change one capacity at a time and recompute throughput.
> - Distinguish a local capacity change from a system-level bottleneck.
> - Explain why a worker outage changes the feasible assignment set.


## Setup

This lab is complete at release: there is no separate student/solution notebook pair in this repository.


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## Baseline: three workers, four tasks


In [2]:
edge_path = joinpath(CHEME5800_L5B_DATA, "Workers-Tasks-Bipartite.edgelist")
baseline_graph = build_sensitivity_graph(edge_path)


MyDirectedBipartiteGraphModel(Dict{Int64, MyGraphNodeModel}(5 => MyGraphNodeModel(5, nothing), 12 => MyGraphNodeModel(12, nothing), 8 => MyGraphNodeModel(8, nothing), 1 => MyGraphNodeModel(1, nothing), 6 => MyGraphNodeModel(6, nothing), 11 => MyGraphNodeModel(11, nothing), 9 => MyGraphNodeModel(9, nothing), 3 => MyGraphNodeModel(3, nothing), 7 => MyGraphNodeModel(7, nothing), 4 => MyGraphNodeModel(4, nothing)…), Dict{Tuple{Int64, Int64}, Number}((4, 5) => 1.0, (1, 2) => 1.0, (8, 12) => 1.0, (3, 7) => 1.0, (2, 5) => 1.0, (6, 10) => 1.0, (1, 3) => 1.0, (12, 13) => 1.0, (4, 6) => 1.0, (5, 9) => 1.0…), Dict{Int64, Set{Int64}}(5 => Set([9]), 12 => Set([13]), 8 => Set([12]), 1 => Set([4, 2, 3]), 6 => Set([10]), 11 => Set([13]), 9 => Set([13]), 3 => Set([5, 6, 7, 8]), 7 => Set([11]), 4 => Set([5, 6, 7, 8])…), Dict(5 => (2, 6), 16 => (5, 9), 20 => (9, 13), 12 => (4, 5), 8 => (3, 5), 17 => (6, 10), 1 => (1, 2), 19 => (8, 12), 22 => (11, 13), 23 => (12, 13)…), #undef, #undef, 1, 13, Dict{Tuple{I

In [3]:
baseline_value, baseline_flow = maximumflow(
    baseline_graph, baseline_graph.nodes[1], baseline_graph.nodes[13];
    algorithm = EdmondsKarpAlgorithm(),
)
baseline_report = validate_sensitivity_flow(baseline_graph, baseline_flow, 1, 13)
@test baseline_value == 3.0
@test baseline_report.valid
(value = baseline_value, valid = baseline_report.valid)


(value = 3.0, valid = true)

### Prediction checkpoint

Before running the next cell: if worker 3 can accept two units from the source, can the downstream network route both units to distinct tasks? Identify the cut that limited the baseline and decide whether that cut has changed.


## Scenario 1: Give worker 3 capacity for a second assignment


In [4]:
expanded_graph = deepcopy(baseline_graph)
expanded_graph.capacity[(1, 3)] = (0.0, 2.0)

expanded_value, expanded_flow = maximumflow(
    expanded_graph, expanded_graph.nodes[1], expanded_graph.nodes[13];
    algorithm = EdmondsKarpAlgorithm(),
)
expanded_report = validate_sensitivity_flow(expanded_graph, expanded_flow, 1, 13)
@test expanded_value == 4.0
@test expanded_report.valid
(value = expanded_value, gain = expanded_value - baseline_value)


(value = 4.0, gain = 1.0)

The source-side cut increases from 3 to 4, and worker 3 has enough distinct outgoing task edges. The fourth task can now be completed.


## Scenario 2: Worker 3 becomes unavailable


In [5]:
outage_graph = deepcopy(baseline_graph)
for edge in keys(outage_graph.capacity)
    if edge[1] == 3
        outage_graph.capacity[edge] = (0.0, 0.0)
    end
end

outage_value, outage_flow = maximumflow(
    outage_graph, outage_graph.nodes[1], outage_graph.nodes[13];
    algorithm = EdmondsKarpAlgorithm(),
)
outage_report = validate_sensitivity_flow(outage_graph, outage_flow, 1, 13)
@test outage_value == 2.0
@test outage_report.valid
(value = outage_value, loss = baseline_value - outage_value)


(value = 2.0, loss = 1.0)

## Compare the interventions


In [6]:
scenario_results = DataFrame(
    scenario = ["baseline", "worker 3 capacity = 2", "worker 3 unavailable"],
    maximum_flow = [baseline_value, expanded_value, outage_value],
    independently_valid = [baseline_report.valid, expanded_report.valid, outage_report.valid],
)
pretty_table(scenario_results)


┌───────────────────────┬──────────────┬─────────────────────┐
│              scenario │ maximum_flow │ independently_valid │
│                String │      Float64 │                Bool │
├───────────────────────┼──────────────┼─────────────────────┤
│              baseline │          3.0 │                true │
│ worker 3 capacity = 2 │          4.0 │                true │
│  worker 3 unavailable │          2.0 │                true │
└───────────────────────┴──────────────┴─────────────────────┘


## Interpretation

- Increasing a capacity helps only if a complete source-to-sink route can use it.
- Removing worker 3 lowers throughput by one because only two source-connected workers remain productive.
- Capacity sensitivity is a network property: inspect cuts and conservation, not just the edited edge.

The next meeting expresses these same conservation and capacity rules as linear constraints.
